# PREGUNTA 01: ETL - Carga y Transformación de Datos

In [ ]:
import pandas as pd
import numpy as np

# 1. EXTRACT
df = pd.read_csv('TransformData/movies_1000.csv')

# 2. TRANSFORM
# Reemplazar '\\N' por NaN
df['Año'] = df['Año'].replace('\\N', np.nan)
df['Género'] = df['Género'].replace('\\N', np.nan)

# Convertir Año a numérico
df['Año'] = pd.to_numeric(df['Año'], errors='coerce')

# Filtrar Año >= 2000 y Rating no nulo
df = df[(df['Año'] >= 2000) & (df['Rating (pro)'].notna())]

# Crear columna Categoría_Rating
def clasificar_rating(rating):
    if rating < 5.0:
        return 'Bajo'
    elif rating < 7.0:
        return 'Medio'
    else:
        return 'Alto'

df['Categoría_Rating'] = df['Rating (pro)'].apply(clasificar_rating)

# 3. LOAD
df.to_csv('TransformData/peliculas_transformadas.csv', index=False)

df.head()

- Se cargó el CSV con `pd.read_csv()` (fase Extract).
- Se reemplazaron los valores `'\\N'` por `NaN` y se convirtió `Año` a numérico (fase Transform).
- Se filtraron películas con `Año >= 2000` y rating válido.
- Se creó la columna `Categoría_Rating` usando una función con `apply()`.
- Se guardó el resultado en `TransformData/peliculas_transformadas.csv` (fase Load).

# PREGUNTA 02: Consultas SQL a Data Warehouse

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('warehouse.db')

# Query 1: Total de visitantes por formato
df_formato = pd.read_sql("""
    SELECT formato, COUNT(*) AS total_visitantes
    FROM ticket_sales
    GROUP BY formato
""", conn)
print("Visitantes por formato:")
print(df_formato)
print()

# Query 2: Precio promedio por género
df_precio_genero = pd.read_sql("""
    SELECT genero, AVG(precio) AS precio_promedio
    FROM ticket_sales
    GROUP BY genero
""", conn)
print("Precio promedio por género:")
print(df_precio_genero)
print()

# Query 3: Top 5 películas por recaudación total
df_top5 = pd.read_sql("""
    SELECT pelicula, SUM(precio) AS recaudacion_total
    FROM ticket_sales
    GROUP BY pelicula
    ORDER BY recaudacion_total DESC
    LIMIT 5
""", conn)
print("Top 5 películas por recaudación:")
print(df_top5)

conn.close()

- Se conectó a `warehouse.db` con `sqlite3.connect()`.
- La primera consulta agrupa por `formato` y cuenta los registros.
- La segunda consulta agrupa por `genero` y calcula el promedio de `precio`.
- La tercera consulta agrupa por `pelicula`, suma `precio`, ordena descendente y limita a 5 filas.

# PREGUNTA 03: Visualización de Datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('CleanData/master_data_final.csv')

# Gráfico 1: Barras - cantidad de visitantes por género
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='genero', hue='genero', palette='Set2', legend=False)
plt.title('Cantidad de Visitantes por Género')
plt.xlabel('Género')
plt.ylabel('Cantidad')
plt.show()

# Gráfico 2: Pie - distribución de formatos
formato_counts = df['formato'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(formato_counts, labels=formato_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribución de Formatos')
plt.show()

# Gráfico 3: Histograma - distribución de precios
plt.figure(figsize=(8, 5))
plt.hist(df['precio'], bins=15, color='skyblue', edgecolor='black')
plt.title('Distribución de Precios de Boletos')
plt.xlabel('Precio')
plt.ylabel('Frecuencia')
plt.show()

- Se cargó `master_data_final.csv` y se usaron `matplotlib` y `seaborn` para los gráficos.
- El gráfico de barras usa `sns.countplot()` para mostrar la cantidad por género.
- El gráfico de pastel usa `plt.pie()` con porcentajes automáticos.
- El histograma usa `plt.hist()` con 15 bins para visualizar la distribución de precios.